# 04 · Function walkthrough: the 12 functions, one at a time

**Purpose.** Run each function of the default/prepay model on its own, with a tiny example you can check by hand, to see exactly what it does before it is used inside the full pool projection (notebook 03).

**How to use.** Run the cells top to bottom (Shift+Enter). Each section has:
- **What it does**: the formula and the idea
- **Example**: a small call with printed output
- **Try it**: a number to change, so you can watch the result move

Order: conversions → amortization → prepayment → credit → projection.

| # | Function | File |
|---|---|---|
| 1 | `calculate_smm` | `src/prepayment.py` |
| 2 | `smm_to_cpr` | `src/prepayment.py` |
| 3 | `cdr_to_mdr` | `src/credit_model.py` |
| 4 | `scheduled_principal` | `src/collateral_projection.py` |
| 5 | `refi_incentive` | `src/prepayment.py` |
| 6 | `calculate_cpr` | `src/prepayment.py` |
| 7 | `mark_to_market_ltv` | `src/credit_model.py` |
| 8 | `calculate_credit_event_rate` | `src/credit_model.py` |
| 9 | `calculate_loss_severity` | `src/credit_model.py` |
| 10 | `modification_loss` | `src/credit_model.py` |
| — | `calculate_prepayment`, `calculate_credit_events` | one-line steps used inside #11 |
| 11 | `project_collateral` | `src/collateral_projection.py` |
| 12 | `run_scenarios` | `src/collateral_projection.py` |

All model parameters are **placeholders** until calibration (the `PARAMS` dicts, printed below).

**What you will see (results in brief).** Each function's example confirms the expected economics: monthly rates compound correctly; a 30-year loan amortizes to exactly $0; lower rates speed up prepayment along an S-curve that burns out over time; falling home prices raise LTV, defaults and severity; and the full STACR pool runs from $19.44bn to $7.85bn by the 2031 call in the base case, with losses well inside Freddie Mac's retained first-loss piece. A **Result** cell after each section explains its output.

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.float_format", "{:,.4f}".format)

from src import prepayment as pp, credit_model as cm, collateral_projection as cp
print("prepayment PARAMS:", pp.PARAMS)
print("credit PARAMS:   ", cm.PARAMS)

### Three toy loans used throughout
Small enough to check by hand: a good borrower, a weaker borrower with a higher coupon, and an investor loan that is 30 days delinquent.

In [ ]:
toy = pd.DataFrame({
    "Loan ID": ["good", "weaker", "investor_dq30"],
    "Current Balance": [300_000.0, 500_000.0, 400_000.0],
    "Gross Coupon": [6.75, 7.25, 7.50],
    "Credit Score": [780, 680, 720],
    "Original LTV": [75, 80, 80],
    "HPI Adjusted LTV": [65.0, 78.0, 75.0],
    "Current Debt to Income": [30, 45, 40],
    "Age": [18, 18, 18],
    "Months to Maturity": [342, 342, 342],
    "Occupancy": ["Owner Occupied", "Owner Occupied", "Investment Property"],
    "dq_bucket": [0, 0, 1],
})
toy

---
## 1 · `calculate_smm`: annual prepay rate → monthly

**What it does.** CPR is the share of the balance prepaid over a year, and SMM is the share prepaid each month. A balance survives a year only if it survives 12 months in a row:

$$1 - CPR = (1 - SMM)^{12} \quad\Rightarrow\quad SMM = 1 - (1 - CPR)^{1/12}$$

That is why SMM is a bit **more** than CPR/12: the monthly rate applies to a shrinking balance.

In [ ]:
cprs = np.array([0.06, 0.10, 0.20, 0.40, 0.60])
pd.DataFrame({"CPR": cprs, "SMM": pp.calculate_smm(cprs), "naive CPR/12": cprs / 12})

**Result #1.** SMM is always a bit **above** CPR/12, and the gap widens with speed: 6% CPR → 0.51% vs 0.50%, 20% → 1.84% vs 1.67%, and 60% → **7.35% vs 5.00%**. Dividing by 12 would understate monthly prepayments by up to a third at high speeds. The monthly rate has to be larger because it applies to a balance that is already shrinking.

**Try it:** what SMM does 100% CPR give? Why does the formula break down there?

---
## 2 · `smm_to_cpr`: monthly → annual (the inverse)

$$CPR = 1 - (1 - SMM)^{12}$$

Useful for reporting: the model works monthly, but the market quotes CPR.

In [ ]:
smm = pp.calculate_smm(cprs)
print("SMM:        ", smm.round(5))
print("back to CPR:", pp.smm_to_cpr(smm).round(5))   # should equal the original CPRs
print("1% SMM is a CPR of", f"{pp.smm_to_cpr(0.01):.2%}")

**Result #2.** The round trip returns exactly 6%, 10%, 20%, 40% and 60%, so the two functions are true inverses. Also, **1% SMM is 11.36% CPR**, not 12%. This is the function used to report the model's monthly prepayments as the CPRs the market quotes.

---
## 3 · `cdr_to_mdr`: annual default rate → monthly

Same math as #1, applied to defaults: $MDR = 1 - (1 - CDR)^{1/12}$. The default model's base rate (0.2% CDR) is set annually and converted here.

In [ ]:
cdrs = np.array([0.002, 0.01, 0.05])
pd.DataFrame({"CDR": cdrs, "MDR": cm.cdr_to_mdr(cdrs)})

**Result #3.** The base default rate of 0.2% CDR becomes a monthly rate of **0.017%** (0.0002 in the table), 1% CDR is 0.084% a month, and 5% CDR is 0.43% a month. At these low levels MDR ≈ CDR/12. The compounding gap only matters at high rates, which is the same math as #1.

---
## 4 · `scheduled_principal`: contractual amortization

**What it does.** A 30-year fixed mortgage has a level payment. Each month:

$$\text{payment} = \frac{B\,r}{1-(1+r)^{-n}}, \qquad \text{interest} = B\,r, \qquad \text{principal} = \text{payment} - \text{interest}$$

with $r$ = coupon / 1200 and $n$ = months remaining. Hand check: $300k, 6.75%, 342 months → **$290.46**.

In [ ]:
print(cp.scheduled_principal(np.array([300_000.0]), np.array([6.75]), np.array([342])))

# Amortize one loan to maturity: principal share grows over time
b, rows = 300_000.0, []
for m in range(342):
    p = cp.scheduled_principal(np.array([b]), np.array([6.75]), np.array([342 - m]))[0]
    rows.append((m + 1, p, b * 0.0675 / 12)); b -= p
amort = pd.DataFrame(rows, columns=["month", "principal", "interest"]).set_index("month")
amort.plot(title="Level-pay mortgage: principal vs interest each month"); plt.ylabel("$")
print("balance after last payment:", round(b, 6))

**Result #4.** Month-1 principal is **$290.46**, matching the hand calculation. The payment is $1,977.96, of which $1,687.50 is interest. The plot shows the level-pay structure: interest dominates early (85% of the first payment), the lines cross around month 220, and the last payment is almost all principal ($1,966.89 vs $11.06 of interest). The balance ends at exactly **$0.00**, so the amortization is correct.

For the pool this means **scheduled principal is small** for young loans, about 0.1% of the balance a month. Most of the pool's principal will come from prepayments (#6).

**Try it:** change the coupon to 3.0. Why does the principal share start much higher?

---
## 5 · `refi_incentive`: how far in the money each loan is

$$\text{incentive}_{t} = \text{loan coupon} - \text{market mortgage rate}_{t} \quad (\text{pp})$$

Positive means the borrower could lower their rate by refinancing. Output shape: (loans × months).

In [ ]:
rates = np.array([7.00, 6.50, 6.25, 6.00, 5.50, 5.00])   # a falling-rate path, 6 months
pd.DataFrame(pp.refi_incentive(toy, rates), index=toy["Loan ID"],
             columns=[f"m{i+1} @ {r}%" for i, r in enumerate(rates)])

**Result #5.** Incentive = coupon − market rate. At 7.00% only the `good` loan (6.75% coupon) is out of the money (**−0.25 pp**). As rates fall to 5.00%, every loan becomes deeply in the money (+1.75 to +2.50 pp). Higher-coupon loans always have more incentive at the same market rate, so in a rally they are the first to refinance.

---
## 6 · `calculate_cpr`: the prepayment S-curve

$$CPR = \underbrace{\min(1, \tfrac{age}{30})}_{\text{seasoning}} \times \Big[\underbrace{6\%}_{\text{turnover}} + \underbrace{55\%}_{\text{refi max}} \times \underbrace{\frac{1}{1+e^{-3.5(\text{inc}-0.75)}}}_{\text{S-curve}} \times \underbrace{e^{-0.02\sum \text{past inc}^+}}_{\text{burnout}}\Big]$$

- **Turnover**: people move even with no rate benefit.
- **S-curve**: little refinancing out of the money, steep around +0.75 pp, saturating deep in the money.
- **Burnout**: loans that stayed in the money without refinancing are the less responsive ones.

In [ ]:
# (a) The S-curve: month-1 CPR vs incentive for one seasoned loan
one = toy.iloc[[0]].assign(Age=60)
grid = np.arange(-2, 3.01, 0.1)
s_curve = [pp.calculate_cpr(one, np.array([one["Gross Coupon"].iloc[0] - x]))[0, 0] for x in grid]
plt.plot(grid, s_curve); plt.axvline(0, color="grey", lw=0.5)
plt.xlabel("refi incentive (pp)"); plt.ylabel("CPR"); plt.title("S-curve (seasoned loan, month 1)")
plt.show()

# (b) Burnout: hold rates 2pp below the coupon for 5 years
path = np.full(60, 6.75 - 2.0)
plt.plot(pp.calculate_cpr(one, path)[0]); plt.xlabel("month"); plt.ylabel("CPR")
plt.title("Burnout: same incentive, falling CPR over time")

In [ ]:
# (c) The three toy loans on the falling-rate path from #5
pd.DataFrame(pp.calculate_cpr(toy, rates), index=toy["Loan ID"],
             columns=[f"m{i+1} @ {r}%" for i, r in enumerate(rates)])

**Result #6.**
- **S-curve (first plot).** For a seasoned loan, CPR is about **6%** when 2 pp out of the money (turnover only), **9.7%** at zero incentive, **33.5%** at the +0.75 pp midpoint, **57%** at +1.5 pp, and it levels off at about **61%**. The steep section between 0 and +1.5 pp is where prepayments are most sensitive to rates. The STACR pool sits there today, with a WA incentive of about +0.5 pp at a 6.25% market rate.
- **Burnout (second plot).** At a constant +2 pp incentive, CPR falls from **60% in month 1 to 41% by month 12 and 11% by month 60**. Borrowers who could have refinanced and did not are the slow ones, so the remaining pool gets slower over time.
- **Toy loans on the falling-rate path (table).** All three speed up as rates fall: `good` goes from **4.8% to 45% CPR**. The investor loan starts fastest because its 7.50% coupon gives the most incentive. By month 6 they converge around **44–45%** because they are near the top of the S-curve. All three are 18 months old, so the 30-month seasoning ramp still scales their CPR to 63% of full speed in month 1, rising to 80% by month 6.

**Try it:** set `pp.PARAMS['midpoint'] = 0.5` and rerun. Where does the steep part move? (Set it back to 0.75 afterwards.)

---
## 7 · `mark_to_market_ltv`: LTV along a house-price path

$$LTV_t = \frac{\text{HPI-adjusted LTV on the tape}}{HPI_t}, \qquad HPI_0 = 1$$

If prices fall 20%, a 65 LTV loan becomes 65 / 0.8 = 81.25. (Amortization is ignored, which is slightly conservative.)

In [ ]:
hpi_down = np.linspace(1.0, 0.80, 13)       # 12 months, -20%
ltv = cm.mark_to_market_ltv(toy, hpi_down)
pd.DataFrame(ltv[:, [0, 5, 11]], index=toy["Loan ID"], columns=["month 1", "month 6", "month 12"])

**Result #7.** A 20% home-price decline over 12 months raises every loan's LTV by a factor of 1/0.8: `good` goes **65 → 81**, `investor_dq30` **75 → 94** and `weaker` **78 → 97.5**, close to being underwater. This is the channel through which house prices drive both defaults (#8) and severity (#9).

---
## 8 · `calculate_credit_event_rate`: monthly default hazard per loan

$$MDR = MDR_{base}\; e^{\,b_{ltv}(LTV_t-80)}\; e^{-0.012(FICO-750)}\; e^{0.02(DTI-38)} \times 1.5_{[\text{investor}]} \;+\; 1\%_{[\text{30 dq, first 12m}]}$$

with $b_{ltv}$ = 0.04 below 80 LTV and 0.10 above (losing equity is what really drives default). Base CDR = 0.2% a year at LTV 80, FICO 750, DTI 38.

In [ ]:
paths = {"HPI +10%": np.linspace(1, 1.10, 13), "flat": np.ones(13), "HPI -20%": hpi_down}
tbl = pd.DataFrame({name: cm.calculate_credit_event_rate(toy, h)[:, -1] * 12 * 100 for name, h in paths.items()},
                   index=toy["Loan ID"])
tbl.columns = [c + " (annualized %, month 12)" for c in tbl.columns]
tbl

**Result #8** (annualized default rates in month 12).
- **Borrower quality matters a lot.** With flat prices, `weaker` (FICO 680, DTI 45, LTV 78) defaults at **0.49%** a year vs **0.07%** for `good` (FICO 780, DTI 30, LTV 65), about 7× higher.
- **Falling prices amplify it.** At HPI −20%, `weaker`'s LTV crosses 80 into the steep part of the LTV term, and its default rate jumps **6×, to 3.07%**. `good` only doubles to 0.13% because it still has equity.
- **Delinquency dominates.** `investor_dq30` is at **12–14%** in every scenario. A loan already 30 days late carries the extra 1%/month roll-to-default hazard for 12 months, which swamps everything else.

All three move the right way: higher HPI → fewer defaults, and lower HPI → more defaults.

---
## 9 · `calculate_loss_severity`: actual loss when a loan is liquidated

STACR is an **actual-loss** deal: the loss is what is really lost on the sale, not a fixed %.

$$\text{loss} = \underbrace{B\,(1 + 12\cdot\tfrac{c}{1200} + 10\%)}_{\text{balance + missed interest + costs}} - \underbrace{\tfrac{B}{LTV}\cdot HPI \cdot (1-20\%)}_{\text{distressed sale proceeds}} - \underbrace{MI}_{\text{only if orig LTV}>80}$$

Severity = loss / B, floored at 0.

In [ ]:
hpi_levels = np.arange(0.6, 1.21, 0.05)
sev = pd.DataFrame({f"{h:.2f}": cm.calculate_loss_severity(toy, h) for h in hpi_levels}, index=toy["Loan ID"]).T
sev.plot(marker="o"); plt.xlabel("HPI at liquidation (today = 1.0)"); plt.ylabel("severity")
plt.title("Severity vs home prices"); plt.show()
sev.loc[["0.80", "1.00", "1.10"]]

**Result #9.** Severity is **0% while the borrower has enough equity**, then rises steeply as prices fall.
- `good` (LTV 65): **0% at today's prices**, only starts losing below about HPI 0.94, and reaches **18%** at HPI 0.80.
- `weaker` (LTV 78): already **15%** at today's prices, because a 20% distressed-sale discount plus 10% costs plus 12 months of missed interest exceed its 22% equity. It reaches **35%** at HPI 0.80.
- `investor_dq30` (LTV 75): 11% today and 32% at HPI 0.80.

This is the actual-loss logic of STACR: equity is the first protection. With the pool's WA HPI-adjusted LTV of about 70, base-case severity is close to zero, and losses appear only in a meaningful price decline.

**Try it:** at what HPI does the *good* loan (LTV 65) start losing money? Compare that with the STACR pool's WA HPI-adjusted LTV of about 70.

---
## 10 · `modification_loss`: cost of rate cuts

Some defaults are modified instead of liquidated (25%, placeholder). The borrower's rate is cut by 2 pp, and under STACR the lost interest is a loss (PPM p. 84-85):

$$\text{monthly mod loss} = \text{modified balance} \times \frac{2.0}{1200}$$

In [ ]:
bal = np.array([100_000.0, 1_000_000.0, 50_000_000.0])
pd.DataFrame({"modified balance": bal, "monthly loss ($)": cm.modification_loss(bal),
              "annual loss ($)": cm.modification_loss(bal) * 12})

**Result #10.** A 2 pp rate cut costs **$166.67 a month per $100k** of modified balance ($2,000 a year), and $1mm costs $1,667 a month. Even $50mm of modified loans only costs **$1mm a year**. Modification losses are small and steady compared with liquidation losses, but they hit the junior tranches' **interest** first (PPM p. 84-85), so they still matter to the waterfall.

---
## 11 · `project_collateral`: everything together for one scenario

Each month, in this order (so a dollar never leaves twice):
1. **Defaults** = MDR × performing balance → into a 12-month liquidation pipeline (25% modified instead)
2. **Scheduled principal** on what is left
3. **Prepayments** = SMM × (balance after scheduled principal)
4. **Credit events** = defaults from 12 months ago leave the pool; **loss** = balance × severity at that month's HPI

$$\text{ending} = \text{beginning} - \text{scheduled} - \text{prepayments} - \text{credit events}$$

First on the toy loans (readable), then on the real STACR pool.

In [ ]:
N = 24
toy_cf = cp.project_collateral(toy, np.linspace(1, 0.85, N + 1), np.full(N, 6.0), N)
toy_cf.round(0).head(15)

In [ ]:
check = toy_cf.beginning_balance - toy_cf.scheduled_principal - toy_cf.prepayments - toy_cf.defaults - toy_cf.ending_balance
print("balance identity, max error ($):", check.abs().max())
print("first credit events in month:", toy_cf.loc[toy_cf.defaults > 0, "month"].min(), "(12-month liquidation lag)")

In [ ]:
pool = pd.read_parquet(ROOT / "data" / "processed" / "stacr_dna1.parquet")
N = 53                                          # today -> Feb 2031 call
cf = cp.project_collateral(pool, np.linspace(1, 1.15, N + 1), np.full(N, 6.25), N)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
(cf.set_index("month")[["scheduled_principal", "prepayments", "defaults"]] / 1e6).plot.area(ax=ax[0])
ax[0].set_ylabel("$mm per month"); ax[0].set_title("Where principal goes (base)")
(cf.set_index("month")["ending_balance"] / 1e9).plot(ax=ax[1])
ax[1].set_ylabel("$bn"); ax[1].set_title("Pool balance (base)")
plt.tight_layout()

**Result #11.**
- **Toy loans, 24 months, HPI −15%, rate 6%.** The $1.2mm starts paying down mostly through **prepayments** (about $37k a month vs about $1k of scheduled principal). Nothing liquidates for the first 12 months, then credit events start in **month 13**, exactly the 12-month liquidation lag. Over 24 months: about $31k of credit events, **$7.1k of losses**, and $216 of modification losses, ending at $486k. The balance identity holds to within **$0.0000000002**.
- **Full STACR pool, base case (plots).** Month 1 has **$283mm of prepayments vs $19mm of scheduled principal**. Prepayments are about **93%** of all principal leaving the pool over the 53 months. Credit events are barely visible on the chart. The pool pays down **60%, from $19.44bn to $7.85bn**, by the Feb 2031 call. The prepay area shrinks over time as the balance falls and burnout slows the remaining loans.

---
## 12 · `run_scenarios`: the same projection on several paths

Takes `{name: {"hpi": path, "rate": path}}` and returns `{name: table}`. These tables are what the waterfall consumes. The paths below are **placeholders** until Coco's scenarios are ready.

In [ ]:
scenarios = {
    "good":     {"hpi": np.linspace(1.00, 1.25, N + 1), "rate": np.full(N, 5.00)},
    "base":     {"hpi": np.linspace(1.00, 1.15, N + 1), "rate": np.full(N, 6.25)},
    "moderate": {"hpi": np.linspace(1.00, 0.90, N + 1), "rate": np.full(N, 5.75)},
    "severe":   {"hpi": np.linspace(1.00, 0.75, N + 1), "rate": np.full(N, 5.25)},
}
results = cp.run_scenarios(pool, scenarios, N)
pd.DataFrame({name: {
    "end balance ($bn)": df.ending_balance.iloc[-1] / 1e9,
    "credit events ($mm)": df.defaults.sum() / 1e6,
    "losses ($mm)": df.losses.sum() / 1e6,
    "loss / cut-off (%)": df.losses.sum() / cp.CUTOFF_BALANCE * 100,
    "avg severity (%)": df.losses.sum() / df.defaults.sum() * 100,
} for name, df in results.items()})

**Result #12** (placeholder scenarios).

| | good | base | moderate | severe |
|---|---|---|---|---|
| End balance | $4.41bn | $7.85bn | $5.26bn | $4.34bn |
| Credit events | $88mm | $101mm | $105mm | $121mm |
| Losses | $2.6mm | $3.7mm | $10.6mm | $21.4mm |
| Loss / cut-off | 0.011% | 0.016% | 0.047% | 0.094% |
| Avg severity | 2.9% | 3.6% | 10.1% | 17.6% |

- **Good** (HPI +25%, rates 5.00%): the fastest paydown and the fewest losses. That's the expected pattern of high CPR, low credit events and low severity. The small loss that remains comes from loans already delinquent on the tape.
- **Losses rise with stress mostly through severity** (2.9% → 17.6%), not through the number of defaults (+38% from good to severe). For this low-LTV pool, home prices drive losses through equity.
- **Stress also pays the pool down faster than base** ($4.34bn vs $7.85bn). That is because the placeholder stress paths also **lower rates**, so many loans refinance away before prices bottom out. This flatters the severe case (see the Try it below).
- **Even severe losses (0.094%) stay inside B-3H (0–0.25%)**, the first-loss piece Freddie Mac retains. The offered A-1, M-1 and M-2 notes take **no write-downs**, and M-2B attaches at 1.90%.

**Try it:** add a scenario `"severe_no_refi"` with HPI to 0.75 but rates at 7.5%. Do losses go up or down vs `severe`, and why? (Hint: who is still in the pool when home prices bottom out?)

---
## Summary of results

| # | Function | What the output showed |
|---|---|---|
| 1–2 | `calculate_smm`, `smm_to_cpr` | 6% CPR = 0.51% SMM, 60% CPR = 7.35% SMM (not 5%); 1% SMM = 11.36% CPR; round trip exact |
| 3 | `cdr_to_mdr` | 0.2% CDR = 0.017% a month; compounding only matters at high rates |
| 4 | `scheduled_principal` | $290.46 in month 1 (85% of the payment is interest); balance hits $0.00 at maturity |
| 5 | `refi_incentive` | from −0.25 to +2.50 pp across the falling-rate path |
| 6 | `calculate_cpr` | S-curve from ~6% (turnover) to ~61%; burnout takes CPR from 60% to 11% over 5 years at constant incentive |
| 7 | `mark_to_market_ltv` | HPI −20% lifts LTV 65 → 81 and 78 → 97.5 |
| 8 | `calculate_credit_event_rate` | weaker borrower ~7× the good one; HPI −20% multiplies it by 6; 30-day dq loan ~12–14% a year |
| 9 | `calculate_loss_severity` | 0% with equity; good loan starts losing below HPI 0.94; 18–35% at HPI 0.80 |
| 10 | `modification_loss` | 2 pp cut = $1,667 a month per $1mm |
| 11 | `project_collateral` | identity holds; first credit events in month 13; ~93% of principal is prepayment; pool −60% by Feb 2031 |
| 12 | `run_scenarios` | losses 0.011% / 0.016% / 0.047% / 0.094% of cut-off (good / base / moderate / severe), all inside B-3H; offered notes untouched |

**Main takeaway.** For this low-LTV STACR pool, the offered notes are exposed mainly to **prepayment speed (timing and WAL)**, not credit loss. Home prices drive losses through severity, and borrower equity absorbs moderate declines.

**Limits to fix next.** (1) All parameters are **placeholders** until calibrated on the Freddie Single-Family Loan-Level Dataset. (2) Underwater borrowers can still refinance in the model, which understates stress losses. (3) The scenario paths are placeholders until Coco's `scenarios.py` is ready.